# Классификация проектов с помощью линейной регрессии

In [3]:
# install libraries

%pip install numpy==1.23.5
%pip install typer==0.9.4
%pip install torch==2.0.1
%pip install transformers==4.34.0
%pip install sentence-transformers==3.0.0
%pip install spacy==3.5.4
%pip install tensorflow==2.12.0
%pip install torchtext==0.15.2
%pip install nltk==3.7
%pip install scipy==1.15.3
%pip install gensim==4.4.0
%pip install xgboost==1.7.6
%pip install catboost

%pip check

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [16]:
# library deps
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from gensim.models.word2vec import Word2Vec
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

## Загрузка данных

In [17]:
Labels = [
    "Автомобильные дороги",
    "Водоотведение",
    "Водопроводы",
    "Газоны дорожки",
    "Газопроводы",
    "Горные выработки",
    "Железнодорожные пути",
    "Заводы фабрики",
    "Здания",
    "Инженерное обеспечение",
    "Инфраструктура наземного электротранспорта",
    "Линии электропередачи",
    "Метрополитены",
    "Мосты и тоннели",
    "Наружное освещение",
    "Нефтепроводы",
    "Сооружения",
    "Теплопроводы",
    "Технологические установки",
]

ColumnNames = ["id", "project_name", "label"]


def load_labeled_data(path):
    labeled_dataframes = [
        pd.read_csv(f"{path}//{label}.csv", names=ColumnNames, header=0)
        for label in tqdm(Labels)
    ]
    result_df = pd.concat(labeled_dataframes)
    result_df["project_name"] = result_df["project_name"].str.strip('"')
    return result_df


def load_unlabeled_data(path):
    return pd.read_csv(path, sep=";", encoding="utf-8", nrows=200000, names=["id", "project_name"])


# raw_df = load_unlabeled_data(f'../Data/Реестр 2022-2024 clean.csv')
# raw_df

## Разделение данных на тестовую и обучающую выборки

In [18]:
def data_train_test_split(data, labels):
    assert len(data) == len(
        labels
    ), "Размеры списков данных и результатов разметки не совпадают"
    le = LabelEncoder()
    le.fit(labels)
    y = le.transform(labels)
    return train_test_split(data, y, test_size=0.2, random_state=42)

## Способы векторизации

In [19]:
def vectorize_words_with_word2vec(sentences, vector_size):
    nltk.download("punkt")
    tokenized_sentences = [
        nltk.tokenize.word_tokenize(text.lower(), language="russian")
        for text in tqdm(sentences)
    ]
    sentence_vectors = Word2Vec(
        tokenized_sentences,
        workers=8,
        vector_size=vector_size,
        min_count=3,
        window=5,
        epochs=15,
    )
    return sentence_vectors


def vectorize_with_word2vec(sentences, vector_size):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.average(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [20]:
# Вернет матрицу размера (len(sentences, 1024)
def vectorize_with_sentence_transformer(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences.to_numpy())

## Функции оптимизации с помощью Grid search и Random search

In [74]:
def find_best_model_gs(X_train, y_train, estimator, param_grid):
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="accuracy",
        cv=3,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_

def find_best_model_rs(X_train, y_train, estimator, param_dist, n_iter):
    random_search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="accuracy",
        cv=3,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_, random_search.best_params_

## Классификация проектов

### Загрузим данные

In [22]:
df = load_labeled_data("../Data/Reestr/Размеченные")
df

100%|██████████| 19/19 [00:00<00:00, 671.28it/s]


,id,project_name,label
0,000352ac-d728-470f-b466-dbaf03df82ad,Строительство автомобильной дороги общего поль...,Автомобильные дороги
1,0004719e-2520-4e50-92ab-b26be1e9ed0e,Капитальный ремонт автомобильной дороги по ул....,Автомобильные дороги
2,000f1480-522e-4e22-94fd-539f1e93a22b,Капитальный ремонт автомобильной дороги общего...,Автомобильные дороги
3,0012f905-46ab-4648-b519-2f1f7e2c2b1a,Капитальный ремонт автомобильной дороги Р-241 ...,Автомобильные дороги
4,001ae5dc-0cb7-40be-b52e-5e33dec04fb7,Реконструкция дорожного покрытия ул. Пограничн...,Автомобильные дороги
...,...,...,...
95,06ee6bfe-d776-474c-8209-c4d666aa947f,"Строительство блока отстойников на УППН ""Сухан...",Технологические установки
96,06f42c68-acde-49a5-9cf5-5cb1e4edd89a,Реконструкция опасного производственного объек...,Технологические установки
97,072313c2-4757-49ab-8e42-0aa2e1a3de12,"Кусты №4Б, №15, №59, №64Б Сыморьяхского местор...",Технологические установки
98,077a2af1-b8af-4ce6-bb0e-e8edde2200f4,Обустройство куста скважин № 407б Тагринского ...,Технологические установки


### Векторизация

In [ ]:
# Векторизация WordToVek
# word2vec_vectors = vectorize_with_word2vec(df['project_name'], 300)

# word2vec_vectors.wv.most_similar('мост')

In [24]:
# Векторизация с помощью SentenceTransformer с использованием модели 'sberbank-ai/sbert_large_nlu_ru'
sbert_vectors = vectorize_with_sentence_transformer(
    "sberbank-ai/sbert_large_nlu_ru", df["project_name"]
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Разделeние на тестовую и обучающую выборки

In [60]:
X_train, X_test, y_train, y_test = data_train_test_split(sbert_vectors, df["label"])

### Логистическая регрессия

#### Параметры модели

In [68]:
log_reg_param_grid = {
        "C": [0.05, 0.1, 1, 10, 50],
        "penalty": ["l1", "l2", 'elasticnet'],
        "solver": ["liblinear", "saga"],
        "max_iter": [500, 1000]}

log_reg_test_param_grid = {
        "C": [1],
        "penalty": ["l1"],
        "solver": ["liblinear"],
        "max_iter": [100]}

 #### Вариант с векторизацией sber sentence transformer

In [ ]:
# Выбор лучшей модели
lr_initial_model = LogisticRegression(random_state=42)
#lr_best_model, lr_best_params = find_best_model_gs(X_train, y_train, lr_initial_model, log_reg_test_param_grid)
lr_best_model, lr_best_params = find_best_model_rs(X_train, y_train, lr_initial_model, log_reg_test_param_grid, 10)
print("Лучшие параметры: ", lr_best_params)

lr_best_model.fit(X_train, y_train)
y_pred = lr_best_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Лучшие параметры:  {'solver': 'liblinear', 'penalty': 'l1', 'max_iter': 100, 'C': 1}


c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Точность: 0.5973684210526315
              precision    recall  f1-score   support

           0       0.83      0.53      0.65        19
           1       0.35      0.39      0.37        18
           2       0.33      0.36      0.35        22
           3       0.75      0.62      0.68        24
           4       0.90      0.39      0.55        23
           5       0.75      0.60      0.67        25
           6       0.84      0.94      0.89        17
           7       0.55      0.69      0.61        16
           8       0.38      0.73      0.50        11
           9       0.50      0.61      0.55        23
          10       0.35      0.73      0.47        11
          11       0.75      0.75      0.75        20
          12       0.64      0.90      0.75        20
          13       0.58      0.64      0.61        22
          14       0.79      0.76      0.78        25
          15       0.54      0.72      0.62        18
          16       0.00      0.00      0.00        2

c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Edu\ML\Diploma\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### XGBoost

#### Параметры модели

In [70]:
xgb_test_param_grid = {"n_estimators": [50], "max_depth": [6], "learning_rate": [1]}

xgb_param_grid = {
    "n_estimators": [50, 80, 100, 150, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.3, 0.5, 1],
}

#### Вариант с векторизацией sber sentence transformer

In [ ]:
xgb_initial_model = XGBClassifier(use_label_encoder=False, n_jobs=-1)
#xgb_best_model, xgb_best_params = find_best_model_gs(X_train, y_train, xgb_initial_model, xgb_test_param_grid)
xgb_best_model, xgb_best_params = find_best_model_rs(X_train, y_train, xgb_initial_model, xgb_test_param_grid, 1)
xgb_best_model.fit(X_train, y_train)
y_pred = xgb_best_model.predict(X_test)

print("Лучшие параметры: ", xgb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

c:\Edu\ML\Diploma\.venv\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


Лучшие параметры:  {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 1}
Точность: 0.6973684210526315
              precision    recall  f1-score   support

           0       0.62      0.84      0.71        19
           1       0.40      0.56      0.47        18
           2       0.50      0.41      0.45        22
           3       0.90      0.79      0.84        24
           4       0.76      0.57      0.65        23
           5       0.63      0.68      0.65        25
           6       1.00      0.82      0.90        17
           7       0.71      0.62      0.67        16
           8       0.50      0.73      0.59        11
           9       0.76      0.83      0.79        23
          10       0.71      0.91      0.80        11
          11       0.62      0.65      0.63        20
          12       0.90      0.90      0.90        20
          13       0.94      0.68      0.79        22
          14       0.76      0.76      0.76        25
          15       0.71      0

### CatBoost

#### Параметры модели

In [ ]:
cb_param_grid = {
        "learning_rate": [0.01, 0.05, 0.1, 0.5],
        "depth": [3, 4, 6, 8],
        "l2_leaf_reg": [1, 2, 4, 8, 10],
        "iterations": [50, 100, 300, 500],
        "random_strength": [0.5, 1, 2.0],
        "bagging_temperature": [0.8, 1.0, 1.2]}

cb_test_param_grid = {
        "learning_rate": [0.1],
        "depth": [6],
        "l2_leaf_reg": [3],
        "iterations": [50],
        "random_strength": [1],
        "bagging_temperature": [1.0]}

#### Вариант с векторизацией sber sentence transformer

In [75]:
cb_initial_model = CatBoostClassifier(iterations=100, loss_function='MultiClass', random_seed=42)
#cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_test_param_grid)
cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_test_param_grid, 1)

#cb_best_model.fit(X_train, y_train)
y_pred = cb_best_model.predict(X_test)

print("Лучшие параметры: ", cb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0:	learn: 2.8318835	total: 2.72s	remaining: 2m 13s
1:	learn: 2.7338311	total: 5.3s	remaining: 2m 7s
2:	learn: 2.6648235	total: 7.76s	remaining: 2m 1s
3:	learn: 2.5817534	total: 10.2s	remaining: 1m 57s
4:	learn: 2.5056912	total: 12.6s	remaining: 1m 53s
5:	learn: 2.4298084	total: 15s	remaining: 1m 50s
6:	learn: 2.3650498	total: 17.5s	remaining: 1m 47s
7:	learn: 2.2966271	total: 19.8s	remaining: 1m 44s
8:	learn: 2.2398341	total: 22.2s	remaining: 1m 41s
9:	learn: 2.1799301	total: 24.6s	remaining: 1m 38s
10:	learn: 2.1190418	total: 27.1s	remaining: 1m 35s
11:	learn: 2.0689308	total: 29.5s	remaining: 1m 33s
12:	learn: 2.0204956	total: 32s	remaining: 1m 31s
13:	learn: 1.9715353	total: 34.6s	remaining: 1m 28s
14:	learn: 1.9227728	total: 37s	remaining: 1m 26s
15:	learn: 1.8741750	total: 39.3s	remaining: 1m 23s
16:	learn: 1.8323328	total: 41.7s	remaining: 1m 20s
17:	learn: 1.7862053	total: 44s	remaining: 1m 18s
18:	learn: 1.7396498	total: 46.4s	remaining: 1m 15s
19:	learn: 1.6916007	total: 48.9s